In [1]:
from pyspark.sql import SparkSession as Spksess,functions as F ,types as T,DataFrame as DFT
from pyspark import SparkConf,StorageLevel, SparkContext
from pyspark.errors import PySparkException as PyEx
from pyspark.sql.utils import AnalysisException as AnyEx
import sys,os,re
from datetime import datetime as dt
from pyspark.sql.functions import col, sum as spark_sum, col, count, when, isnan, isnull
from pyspark.sql.types import *
import pandas as pd 
import traceback

In [2]:
sparkconfiguration  = SparkConf()

In [3]:
sparkconfiguration.set("spark.app.name","John_MicrosoftSparkJob-ML") 
sparkconfiguration.set("spark.master", "local[*]")                
sparkconfiguration.set("spark.driver.memory", "4g")      
sparkconfiguration.set("spark.driver.cores", "1")                 
sparkconfiguration.set("spark.ui.port", "4040")                
sparkconfiguration.set("spark.executor.memory", "2g")      
sparkconfiguration.set("spark.executor.cores", "2")               
sparkconfiguration.set("spark.executor.instances", "3")         
sparkconfiguration.set("spark.default.parallelism","27")         
sparkconfiguration.set("spark.sql.shuffle.partitions","30")       
sparkconfiguration.set("spark.task.cpus", "1")  
sparkconfiguration.set("spark.memory.fraction","0.8")            
sparkconfiguration.set("spark.memory.storageFraction","0.5")     
sparkconfiguration.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")  
sparkconfiguration.set("spark.kryo.registrationRequired","false")  
sparkconfiguration.set("spark.kryo.classesToRegister", "org.apache.spark.sql.Row")  
sparkconfiguration.set("spark.eventLog.enabled", "true")           
sparkconfiguration.set("spark.eventLog.dir", "/home/john/spark_event_logs/")  
sparkconfiguration.set("spark.history.fs.logDirectory","/home/john/spark_history_logs/")  
sparkconfiguration.set("spark.jars", "/home/john/spark_jar_files/postgresql-42.7.7.jar")

In [4]:
SparkSession_init = Spksess.builder.config(conf=sparkconfiguration).getOrCreate()

25/07/22 17:58:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
database_config_url      = "jdbc:postgresql://{}:{}/{}".format('host.docker.internal', '5432', 'dba_microsoft_cbs_da')
db_credential_properties = {"user": 'john_user',"password": 'abc@12345',"driver": 'org.postgresql.Driver'}

In [6]:
def capture_spark_error(func):
    """
    Decorator function to capture detailed error information from Spark Operations and general exceptions.
    """
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except (AnyEx, PyEx) as e:
            error_info = {
                'success': False,
                'error_type': type(e).__name__,
                'error_message': str(e),
                'error_args': e.args,
                'timestamp': dt.now().isoformat()
            }
            exc_type, exc_value, exc_traceback = sys.exc_info()
            error_info.update({
                'traceback': traceback.format_exc(),
                'exception_type': exc_type,
                'exception_value': exc_value,
                'line_number': exc_traceback.tb_lineno if exc_traceback else None
            })
            return error_info
        except Exception as e:
            # Catch any other exception
            error_info = {
                'success': False,
                'error_type': type(e).__name__,
                'error_message': str(e),
                'error_args': e.args,
                'timestamp': dt.now().isoformat()
            }
            exc_type, exc_value, exc_traceback = sys.exc_info()
            error_info.update({
                'traceback': traceback.format_exc(),
                'exception_type': exc_type,
                'exception_value': exc_value,
                'line_number': exc_traceback.tb_lineno if exc_traceback else None
            })
            return error_info
    return wrapper

In [7]:
absolute_folder_path = "/home/john/program_data"
files = [os.path.join(absolute_folder_path, f)for f in os.listdir(absolute_folder_path) if os.path.isfile(os.path.join(absolute_folder_path, f))]

In [8]:
@capture_spark_error
def read_csv_files(file_path: str) -> DFT:
    df = SparkSession_init.read.option("header", "true").option("inferSchema", "true").csv(file_path)
    return df


In [9]:
test_input_Data = read_csv_files(files[0])
train_input_Data = read_csv_files(files[1])


In [10]:
print(test_input_Data.dtypes)
print(train_input_Data.dtypes)

[('Id', 'bigint'), ('OrgId', 'int'), ('IncidentId', 'int'), ('AlertId', 'int'), ('Timestamp', 'timestamp'), ('DetectorId', 'int'), ('AlertTitle', 'int'), ('Category', 'string'), ('MitreTechniques', 'string'), ('IncidentGrade', 'string'), ('ActionGrouped', 'string'), ('ActionGranular', 'string'), ('EntityType', 'string'), ('EvidenceRole', 'string'), ('DeviceId', 'int'), ('Sha256', 'int'), ('IpAddress', 'int'), ('Url', 'int'), ('AccountSid', 'int'), ('AccountUpn', 'int'), ('AccountObjectId', 'int'), ('AccountName', 'int'), ('DeviceName', 'int'), ('NetworkMessageId', 'int'), ('EmailClusterId', 'double'), ('RegistryKey', 'int'), ('RegistryValueName', 'int'), ('RegistryValueData', 'int'), ('ApplicationId', 'int'), ('ApplicationName', 'int'), ('OAuthApplicationId', 'int'), ('ThreatFamily', 'string'), ('FileName', 'int'), ('FolderPath', 'int'), ('ResourceIdName', 'int'), ('ResourceType', 'string'), ('Roles', 'string'), ('OSFamily', 'int'), ('OSVersion', 'int'), ('AntispamDirection', 'string')

In [11]:
test_missing_columns = set(test_input_Data.columns) - set(train_input_Data.columns)
train_missing_columns = set(train_input_Data.columns) - set(test_input_Data.columns)
print("Missing columns in test:",test_missing_columns )
print("Missing columns in train:",train_missing_columns)


Missing columns in test: {'Usage'}
Missing columns in train: set()


In [12]:

test_input_Data[['Usage']].groupBy('Usage').count().show()


+-------+-------+
|  Usage|  count|
+-------+-------+
|Private|1232555|
| Public|2915437|
+-------+-------+



In [13]:
@capture_spark_error
def  append_missing_columns_dft (input_df: DFT, missing_columns: set, comparison_dft: DFT) -> DFT:
    for column in missing_columns:
        comparision_dtypes = comparison_dft.schema[column].dataType if column in comparison_dft.columns else T.StringType()
        input_df = input_df.withColumn(column, F.lit(None).cast(comparision_dtypes))
    return input_df

In [14]:
test_missing_columns_udft = append_missing_columns_dft(test_input_Data,train_missing_columns, train_input_Data)
train_missing_columns_udft = append_missing_columns_dft(train_input_Data, test_missing_columns, test_input_Data)
print("Missing columns in test:",set(test_missing_columns_udft.columns) - set(train_missing_columns_udft.columns) )
print("Missing columns in train:",set(train_missing_columns_udft.columns) - set(test_missing_columns_udft.columns))

Missing columns in test: set()
Missing columns in train: set()


In [15]:
@capture_spark_error
def count_of_nulls_column(df):
    # Get null counts as a dictionary
    null_counts_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
    data = [(col_name, null_count,str(null_count/1000)+'k') for col_name, null_count in null_counts_row.items() if null_count > 0 ]
    # Create a new DataFrame with two columns: column_name and null_count
    return SparkSession_init.createDataFrame(data, ["column_name", "null_count", "null_countink"])

# Example usage:


In [16]:
test_nulls_columsn_list  = count_of_nulls_column(test_missing_columns_udft)
test_nulls_columsn_list.show(test_nulls_columsn_list.count(),truncate=False)

25/07/15 21:14:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------+----------+-------------+
|column_name      |null_count|null_countink|
+-----------------+----------+-------------+
|MitreTechniques  |2307104   |2307.104k    |
|ActionGrouped    |4146079   |4146.079k    |
|ActionGranular   |4146079   |4146.079k    |
|EmailClusterId   |4106285   |4106.285k    |
|ThreatFamily     |4116614   |4116.614k    |
|ResourceType     |4144998   |4144.998k    |
|Roles            |4039317   |4039.317k    |
|AntispamDirection|4071481   |4071.481k    |
|SuspicionLevel   |3498157   |3498.157k    |
|LastVerdict      |3155260   |3155.26k     |
+-----------------+----------+-------------+



In [17]:
train_nulls_columsn_list  = count_of_nulls_column(train_missing_columns_udft)
train_nulls_columsn_list.show(train_nulls_columsn_list.count(),truncate=False)

+-----------------+----------+-------------+
|column_name      |null_count|null_countink|
+-----------------+----------+-------------+
|MitreTechniques  |5468386   |5468.386k    |
|IncidentGrade    |51340     |51.34k       |
|ActionGrouped    |9460773   |9460.773k    |
|ActionGranular   |9460773   |9460.773k    |
|EmailClusterId   |9420025   |9420.025k    |
|ThreatFamily     |9441956   |9441.956k    |
|ResourceType     |9509762   |9509.762k    |
|Roles            |9298686   |9298.686k    |
|AntispamDirection|9339535   |9339.535k    |
|SuspicionLevel   |8072708   |8072.708k    |
|LastVerdict      |7282572   |7282.572k    |
|Usage            |9516837   |9516.837k    |
+-----------------+----------+-------------+



In [18]:
# Perform a full outer join on 'column_name'
nulls_comparison = train_nulls_columsn_list.alias("train").join(
    test_nulls_columsn_list.alias("test"),
    on="column_name",
    how="full_outer"
).select(
    "column_name",
    F.coalesce(F.col("train.null_count"),F.lit(0)).alias("train_null_count"),
    F.coalesce(F.col("test.null_count"),F.lit(0)).alias("test_null_count")
)


# Add a column to indicate which DataFrame has more nulls
nulls_comparison = nulls_comparison.withColumn(
    "more_nulls_in",
    F.when(
        F.col("train_null_count") > F.col("test_null_count"), F.lit("train")
    ).when(
        F.col("test_null_count") > F.col("train_null_count"), F.lit("test")
    ).otherwise(F.lit("equal"))
)

nulls_comparison.show(nulls_comparison.count(), truncate=False)

+-----------------+----------------+---------------+-------------+
|column_name      |train_null_count|test_null_count|more_nulls_in|
+-----------------+----------------+---------------+-------------+
|ActionGranular   |9460773         |4146079        |train        |
|ActionGrouped    |9460773         |4146079        |train        |
|AntispamDirection|9339535         |4071481        |train        |
|EmailClusterId   |9420025         |4106285        |train        |
|IncidentGrade    |51340           |0              |train        |
|LastVerdict      |7282572         |3155260        |train        |
|MitreTechniques  |5468386         |2307104        |train        |
|ResourceType     |9509762         |4144998        |train        |
|Roles            |9298686         |4039317        |train        |
|SuspicionLevel   |8072708         |3498157        |train        |
|ThreatFamily     |9441956         |4116614        |train        |
|Usage            |9516837         |0              |train     

In [48]:
combined_df[['Usage']].groupBy('Usage').count().show()


25/07/09 22:17:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/09 22:17:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/09 22:17:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/09 22:17:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------+
|  Usage|  count|
+-------+-------+
|Private|5990973|
| Public|7673856|
+-------+-------+



In [19]:
updated_combined_dft = train_missing_columns_udft.unionByName(test_missing_columns_udft)
repartition_count = int(3458 / 128)  
updated_combined_dft.repartition(repartition_count)
updated_combined_dft[['Usage']].groupBy('Usage').count().show()

+-------+-------+
|  Usage|  count|
+-------+-------+
|   NULL|9516837|
|Private|1232555|
| Public|2915437|
+-------+-------+



In [20]:
# Step 1: Read the table into a DataFrame
duplicate_table_df = SparkSession_init.read.jdbc(
    url=database_config_url,
    table="staging_input_data.microsoft_cyber_duplicates",  # replace with your table name
    properties=db_credential_properties
)

# Step 2: Get all columns from the new DataFrame
join_columns = duplicate_table_df.columns



In [21]:
from pyspark.sql.functions import coalesce, lit, col,date_format

# Set aliases
combined_df_alias = updated_combined_dft.alias("a")
duplicate_table_df_alias = duplicate_table_df.alias("b")

def get_coalesce_expr( df, colname,alias= None):
    column = f"{alias}.{colname}"if alias is not None else colname
    dtype = dict(df.dtypes)[colname]
    if dtype in ['int', 'bigint', 'double', 'float', 'decimal']:
        return coalesce(col(column), lit(0))
    elif dtype in ['string']:
        return coalesce(col(column), lit(''))
    elif dtype in ['date', 'timestamp']:
        return coalesce(
            date_format(col(column), "ddMMyyyyHHmmss").cast("long"),
            lit(1010001000000).cast("long")
        )
    else:
        return col(column)

# Build join condition using aliases
join_cond = [
    get_coalesce_expr( updated_combined_dft, c,"a") == get_coalesce_expr(duplicate_table_df, c,"b")
    for c in join_columns if not(c == 'count_value')
]

select_columns = [ col(f"{'a'}.{c}").alias(c)   for c in combined_df_alias.columns]

# Perform anti-join
result_df_without_duplicates = combined_df_alias.join(duplicate_table_df_alias, on=join_cond, how='left_anti')
result_df_with_duplicates = combined_df_alias.join(duplicate_table_df_alias, on=join_cond, how='inner')

# Now select only left table columns (from alias 'a')
result_df_with_duplicates = result_df_with_duplicates.select(
    [col(f"a.{c}").alias(c) for c in combined_df_alias.columns]
)

result_df_without_duplicates = combined_df_alias.join(duplicate_table_df_alias, on=join_cond, how='left_anti')
# Now select only left table columns (from alias 'a')
result_df_without_duplicates = result_df_without_duplicates.select(
    [col(f"a.{c}").alias(c) for c in combined_df_alias.columns]
)

In [22]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

order_by_columns  = [get_coalesce_expr(result_df_with_duplicates, c) for c in result_df_with_duplicates.columns]

# Add a row_number column
dup_dft = result_df_with_duplicates.withColumn("dup_rank", row_number().over(Window.partitionBy(["Id","OrgId","IncidentId","AlertId","AccountSid","AccountObjectId","AccountName","NetworkMessageId","ApplicationId"]).orderBy(order_by_columns)))
dup_dft.repartition(repartition_count)
# # Filter to keep only records with rank == 1
deduped_dft = dup_dft.filter(col("dup_rank") == 1).drop("rank")
deduped_dft = deduped_dft.drop("dup_rank")
deduped_dft.repartition(repartition_count)
duplicate_fixed_dft = result_df_without_duplicates.unionByName(deduped_dft, allowMissingColumns=True)
duplicate_fixed_dft

DataFrame[Id: bigint, OrgId: int, IncidentId: int, AlertId: int, Timestamp: timestamp, DetectorId: int, AlertTitle: int, Category: string, MitreTechniques: string, IncidentGrade: string, ActionGrouped: string, ActionGranular: string, EntityType: string, EvidenceRole: string, DeviceId: int, Sha256: int, IpAddress: int, Url: int, AccountSid: int, AccountUpn: int, AccountObjectId: int, AccountName: int, DeviceName: int, NetworkMessageId: int, EmailClusterId: double, RegistryKey: int, RegistryValueName: int, RegistryValueData: int, ApplicationId: int, ApplicationName: int, OAuthApplicationId: int, ThreatFamily: string, FileName: int, FolderPath: int, ResourceIdName: int, ResourceType: string, Roles: string, OSFamily: int, OSVersion: int, AntispamDirection: string, SuspicionLevel: string, LastVerdict: string, CountryCode: int, State: int, City: int, Usage: string]

In [25]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, when

w = Window().orderBy(F.monotonically_increasing_id())
duplicate_fixed_dft = duplicate_fixed_dft.withColumn("row_num", row_number().over(w))
duplicate_fixed_dft_up = duplicate_fixed_dft.withColumn(
    "Usage",
    when((F.col("row_num") % 2 == 1), "Public").otherwise("Private")
).drop("row_num")

In [27]:
duplicate_fixed_dft_up.write.jdbc(
    url=database_config_url,
    table="staging_input_data.microsoft_cyber_dups",  # replace with your table name
    properties=db_credential_properties,
    mode="overwrite"  # or "append" based on your requirement
)

25/07/15 22:11:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 22:11:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 2

In [35]:
duplicate_fixed_dft_up.drop("MitreTechniques")

rename_dict = {
"\"Id\"":"id",
"\"OrgId\"":"org_id",
"\"IncidentId\"":"incident_id",
"\"AlertId\"":"alert_id",
"\"Timestamp\"":"creation_timestamp",
"\"DetectorId\"":"detector_id",
"\"AlertTitle\"":"alert_title",
"\"Category\"":"category",
"\"IncidentGrade\"":"incident_grade",
"\"ActionGrouped\"":"action_grouped",
"\"ActionGranular\"":"action_granular",
"\"EntityType\"":"entity_type",
"\"EvidenceRole\"":"evidence_role",
"\"DeviceId\"":"device_id",
"\"Sha256\"":"sha256",
"\"IpAddress\"":"ip_address",
"\"Url\"":"url",
"\"AccountSid\"":"account_sid",
"\"AccountUpn\"":"account_upn",
"\"AccountObjectId\"":"account_object_id",
"\"AccountName\"":"account_name",
"\"DeviceName\"":"device_name",
"\"NetworkMessageId\"":"network_message_id",
"\"EmailClusterId\"":"email_cluster_id",
"\"RegistryKey\"":"registry_key",
"\"RegistryValueName\"":"registry_value_name",
"\"RegistryValueData\"":"registry_value_data",
"\"ApplicationId\"":"application_id",
"\"ApplicationName\"":"application_name",
"\"OAuthApplicationId\"":"oauth_application_id",
"\"ThreatFamily\"":"threat_family",
"\"FileName\"":"file_name",
"\"FolderPath\"":"folder_path",
"\"ResourceIdName\"":"resource_id_name",
"\"ResourceType\"":"resource_type",
"\"Roles\"":"roles",
"\"OSFamily\"":"os_family",
"\"OSVersion\"":"os_version",
"\"AntispamDirection\"":"anti_spam_direction",
"\"SuspicionLevel\"":"suspicion_level",
"\"LastVerdict\"":"last_verdict",
"\"CountryCode\"":"country_code",
"\"State\"":"state",
"\"City\"":"city",
"\"Usage\"":"usage",
}
final_microsft_data = duplicate_fixed_dft_up

final_microsft_data = final_microsft_data.select(
    *[col(c).alias(rename_dict.get(f'"{c}"', c)) for c in final_microsft_data.columns]
)
repartition_count = int(final_microsft_data.count() / 128) if final_microsft_data.count() > 128 else 1
final_microsft_data.repartition(repartition_count)


DataFrame[id: bigint, org_id: int, incident_id: int, alert_id: int, creation_timestamp: timestamp, detector_id: int, alert_title: int, category: string, MitreTechniques: string, incident_grade: string, action_grouped: string, action_granular: string, entity_type: string, evidence_role: string, device_id: int, sha256: int, ip_address: int, url: int, account_sid: int, account_upn: int, account_object_id: int, account_name: int, device_name: int, network_message_id: int, email_cluster_id: double, registry_key: int, registry_value_name: int, registry_value_data: int, application_id: int, application_name: int, oauth_application_id: int, threat_family: string, file_name: int, folder_path: int, resource_id_name: int, resource_type: string, roles: string, os_family: int, os_version: int, anti_spam_direction: string, suspicion_level: string, last_verdict: string, country_code: int, state: int, city: int, usage: string]

In [36]:
final_microsft_data.write.jdbc(
    url=database_config_url,
    table="staging_input_data.microsoft_cyber_data",  # replace with your table name
    properties=db_credential_properties,
    mode="overwrite"  )

25/07/15 23:05:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:05:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:05:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:06:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:06:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 23:06:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/15 2

In [8]:
ML_data_input =  SparkSession_init.read.jdbc(
    url=database_config_url,
    table="staging_input_data.microsoft_cyber_data",  # replace with your table name
    properties=db_credential_properties
)

In [10]:
repartition_count = int(3458 / 128)  
ML_data_input.repartition(repartition_count)
ML_data_input = ML_data_input.drop("mitre_techniques")


In [20]:
@capture_spark_error
class EnterpriseDataProfiler() :
    def __init__(self, spark_session):
        self.spark = spark_session

    def get_basic_statistics(self,df:DFT):
        """Get basic dataset statistics"""
        return {
            'total_rows': df.count(),
            'total_columns': len(df.columns),
            'memory_usage_gb': df.cache().count() * len(df.columns) * 8 / (1024**3),  
            'schema': df.schema
        }
    
    def infer_column_category(self, sample_values, spark_type):
        print("Infering column category for spark type:", spark_type)
        print("Sample values:", sample_values[0])

        """Infer the actual category of data"""
        if not sample_values:
            return 'empty'
            
        # Check for timestamp patterns
        timestamp_patterns = ['timestamp', 'date', 'time']
        if any(pattern in str(sample_values[0]).lower() for pattern in timestamp_patterns):
            print("Detected timestamp-like data in sample values.")
            return 'temporal'
        

        # Check if it's actually categorical despite being string
        if len(set(sample_values)) / len(sample_values) < 0.1:  # Less than 10% unique
            return 'categorical'
            
        # Check for numeric data stored as strings
        if spark_type == 'StringType()':
            try:
                numeric_count = sum(1 for val in sample_values[:100] 
                                 if str(val).replace('.', '').replace('-', '').isdigit())
                if numeric_count / len(sample_values[:100]) > 0.8:
                    return 'numeric_as_string'
            except:
                pass   
                
        # Standard categorization
        if 'Int' in spark_type or 'Double' in spark_type or 'Float' in spark_type:
            return 'numerical'
        elif 'String' in spark_type:
            return 'text'
        elif 'Boolean' in spark_type:
            return 'boolean'
        else:
            return 'other'

    def analyze_column_types(self,df:DFT):
        """Automatically categorize columns by their actual content"""
        column_analysis = {}
        
        for field in df.schema.fields:
           
            col_name = field.name
            col_type = str(field.dataType)
            
            # Sample data for content analysis
            sample_data = df.select(col_name).filter(col(col_name).isNotNull()).limit(1000).collect()
            sample_values = [row[0] for row in sample_data if row[0] is not None]
            analysis = {
                'spark_data_type': col_type,
                'inferred_category': self.infer_column_category(sample_values, col_type),
                'sample_values': sample_values[:10],
                'unique_count': df.select(col_name).distinct().count(),
                'null_percentage': self.analyze_missing_data(df)[col_name]['null_percentage'],
                'null_count':self.analyze_missing_data(df)[col_name]['null_count'] 
            }
            
            column_analysis[col_name] = analysis
            
        return column_analysis
        
    def analyze_missing_data(self,df:DFT):
        """Comprehensive missing data analysis"""
        missing_analysis = {}
        
        for col_name in df.columns:
            null_count = df.filter(col(col_name).isNull() | 
                                 (col(col_name) == "") | 
                                 (col(col_name) == "null") |
                                 (col(col_name) == "NULL") |
                                 (col(col_name) == "None")).count()
            
            total_count = df.count()
            null_percentage = (null_count / total_count) * 100
            
            missing_analysis[col_name] = {
                'null_count': null_count,
                'null_percentage': null_percentage,
                'missing_severity': self.classify_missing_severity(null_percentage)
            }
            
        return missing_analysis
    
    def classify_missing_severity(self, null_percentage):
        """Classify missing data severity"""
        if null_percentage == 0:
            return 'none'
        elif null_percentage < 5:
            return 'low'
        elif null_percentage < 20:
            return 'moderate'
        elif null_percentage < 50:
            return 'high'
        else:
            return 'critical'
    
    def calculate_data_quality_score(self,df:DFT):
        """Calculate overall data quality score"""
        null_analysis = self.analyze_missing_data(df)
        
        # Simple scoring based on completeness
        total_cells = df.count() * len(df.columns)
        total_nulls = sum(analysis['null_count'] for analysis in null_analysis.values())
        
        completeness_score = ((total_cells - total_nulls) / total_cells) * 100
        
        return {
            'completeness_score': completeness_score,
            'overall_grade': self.get_quality_grade(completeness_score)
        }
    
    def get_quality_grade(self, score):
        """Convert score to grade"""
        if score >= 95:
            return 'A'
        elif score >= 85:
            return 'B'
        elif score >= 75:
            return 'C'
        elif score >= 65:
            return 'D'
        else:
            return 'F'

    def comprehensive_data_profile(self,df:DFT):
        """
        Comprehensive data profiling to understand dataset characteristics
        """
        profile_results = {
            'basic_stats': self.get_basic_statistics(df),
            'column_types': self.analyze_column_types(df),
            'null_analysis': self.analyze_missing_data(df),
            'data_quality_score': self.calculate_data_quality_score(df),
            'recommendations': []
        }
        
        return profile_results

In [39]:
# SparkSession_init.stop()
SparkSession_init.stop()